# LeetCode 1698 — Number of Distinct Substrings in a String bằng Python

Link đề: https://leetcode.com/problems/number-of-distinct-substrings-in-a-string/ · Độ khó: **Medium** · Chủ đề: String, Trie, Suffix Array, Hash Function, Rolling Hash

> **Bài Premium.** Đề bài trong notebook được viết lại theo hiểu biết về bài (nguồn: bản mô tả của doocs/leetcode). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc trên LeetCode.

Skeleton LeetCode:

```python
class Solution:
    def countDistinct(self, s: str) -> int:
        ...
```

Lộ trình: **set mọi chuỗi con** (`O(n³)`) → **Trie các hậu tố** (`O(n²)`, đếm số nút) → **băm lăn (rolling hash)** theo từng độ dài (`O(n²)`) → **mảng hậu tố + LCP** (`n(n+1)/2 − ΣLCP`) → **suffix automaton** (`O(n)`, trả lời follow-up).

## 1. Mục tiêu học

1. Đếm chuỗi con phân biệt bằng nhận xét nền tảng: **mỗi chuỗi con là tiền tố của một hậu tố** ⇒ Trie của mọi hậu tố có đúng một nút cho mỗi chuỗi con phân biệt.
2. Cài **rolling hash** (đa thức theo modulo) để so sánh chuỗi con trong `O(1)` và hiểu rủi ro **va chạm băm**.
3. Dựng **mảng hậu tố (suffix array)** và **mảng LCP** (thuật toán Kasai), dùng công thức `n(n+1)/2 − Σ LCP`.
4. Biết tới **suffix automaton** — cấu trúc `O(n)` trả lời follow-up — và công thức `Σ (len[v] − len[link[v]])`.
5. Kiểm chứng chéo nhiều thuật toán phức tạp bằng một vét cạn đơn giản trên chuỗi ngẫu nhiên.

## 2. Đề bài & ví dụ

Cho chuỗi `s`, trả về **số chuỗi con phân biệt** của `s`.

Một chuỗi con của `s` thu được bằng cách xóa một số ký tự (có thể 0) ở **đầu** và một số ký tự (có thể 0) ở **cuối** chuỗi — tức là một đoạn liên tiếp **khác rỗng** `s[i..j]` (theo các ví dụ, chuỗi rỗng không được tính).

**Ví dụ 1:** `s = "aabbaba"` → `21`. Tập chuỗi con phân biệt: `a, b, aa, bb, ab, ba, aab, abb, bab, bba, aba, aabb, abba, bbab, baba, aabba, abbab, bbaba, aabbab, abbaba, aabbaba`.

**Ví dụ 2:** `s = "abcdefg"` → `28` (mọi ký tự khác nhau ⇒ mọi đoạn đều khác nhau: `7·8/2 = 28`).

**Ràng buộc:**
- `1 ≤ s.length ≤ 500`
- `s` gồm chữ cái tiếng Anh thường.

**Follow-up:** giải trong `O(n)`?

In [1]:
import random
import time
from typing import List

examples = [
    ("aabbaba", 21),
    ("abcdefg", 28),
]
for s, want in examples:
    print(f"s={s!r} -> kỳ vọng {want}")

s='aabbaba' -> kỳ vọng 21
s='abcdefg' -> kỳ vọng 28


## 3. Kiến thức nền

### 3.1 Liệt kê chuỗi con bằng slicing

Có `n(n+1)/2` cặp `(i, j)` với `0 ≤ i < j ≤ n`, ứng với `s[i:j]`. Nhiều cặp có thể cho **cùng một** chuỗi.

In [2]:
s = "aab"
subs = [s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1)]
print("mọi chuỗi con (có lặp):", subs, "->", len(subs))
print("phân biệt:", sorted(set(subs)), "->", len(set(subs)))

mọi chuỗi con (có lặp): ['a', 'aa', 'aab', 'a', 'ab', 'b'] -> 6
phân biệt: ['a', 'aa', 'aab', 'ab', 'b'] -> 5


### 3.2 Hậu tố và tiền tố

Hậu tố thứ `i` là `s[i:]`. **Mọi chuỗi con `s[i:j]` là một tiền tố của hậu tố `s[i:]`.** Đây là chìa khóa cho Trie, mảng hậu tố và automaton.

In [3]:
s = "aabbaba"
for i in range(len(s)):
    print(i, s[i:])

0 aabbaba
1 abbaba
2 bbaba
3 baba
4 aba
5 ba
6 a


### 3.3 Trie bằng từ điển lồng nhau

Mỗi nút là một `dict` ký tự → nút con. Chèn một chuỗi: đi từ gốc, tạo nút con nếu chưa có. Số nút (trừ gốc) = số **tiền tố phân biệt** của các chuỗi đã chèn.

In [4]:
def trie_count_prefixes(words):
    root, nodes = {}, 0
    for w in words:
        cur = root
        for ch in w:
            if ch not in cur:
                cur[ch] = {}
                nodes += 1
            cur = cur[ch]
    return nodes

print(trie_count_prefixes(["abc", "abd", "b"]), "(a, ab, abc, abd, b)")

5 (a, ab, abc, abd, b)


### 3.4 Băm đa thức (polynomial hash)

`H(s[0..L−1]) = (s0·B^{L−1} + s1·B^{L−2} + … + s_{L−1}) mod M`. Với mảng tiền tố `h[i]` = hash của `s[:i]` và `p[i] = B^i mod M`, hash của `s[i:j]` là `(h[j] − h[i]·p[j−i]) mod M` — tính trong `O(1)`. Chọn `M = 2⁶¹ − 1` (số nguyên tố Mersenne) và `B` ngẫu nhiên để giảm va chạm.

In [5]:
MOD = (1 << 61) - 1
rng = random.Random(12345)
BASE = rng.randrange(256, MOD - 1)

def prefix_hash(s):
    n = len(s); h = [0] * (n + 1); p = [1] * (n + 1)
    for i, ch in enumerate(s):
        h[i + 1] = (h[i] * BASE + ord(ch)) % MOD
        p[i + 1] = p[i] * BASE % MOD
    return h, p

def sub_hash(h, p, i, j):
    return (h[j] - h[i] * p[j - i]) % MOD

h, p = prefix_hash("abcab")
print("hash('ab') ở vị trí 0 và 3 bằng nhau?", sub_hash(h, p, 0, 2) == sub_hash(h, p, 3, 5))
print("hash('ab') vs hash('bc'):", sub_hash(h, p, 0, 2) == sub_hash(h, p, 1, 3))

hash('ab') ở vị trí 0 và 3 bằng nhau? True
hash('ab') vs hash('bc'): False


### 3.5 LCP (longest common prefix) của hai chuỗi

In [6]:
def lcp(a, b):
    k = 0
    while k < len(a) and k < len(b) and a[k] == b[k]:
        k += 1
    return k

print(lcp("abba", "abbab"), lcp("ba", "bab"), lcp("abc", "x"))

4 2 0


## 4. Ý tưởng nền tảng / trực giác

Ví dụ nhỏ `s = "aab"`: các hậu tố là `"aab"`, `"ab"`, `"b"`. Chèn lần lượt vào Trie:

| chèn | nút mới tạo | lý do |
|---|---|---|
| `aab` | `a`, `aa`, `aab` | Trie rỗng |
| `ab` | `ab` | `a` đã có, chỉ tạo nút `ab` |
| `b` | `b` | mới |

Tổng 5 nút = 5 chuỗi con phân biệt (`a, aa, aab, ab, b`). Mỗi lần đi qua một nút **đã có** nghĩa là chuỗi con đó đã được đếm trước đó.

Nếu **sắp xếp** các hậu tố: `aab`, `ab`, `b` — hậu tố liền kề chung tiền tố dài `LCP = 1` (`aab`/`ab` chung `a`) và `0`. Tổng số tiền tố của các hậu tố là `3 + 2 + 1 = 6`; bớt phần trùng `1 + 0 = 1` ⇒ `5`. Đây là ý tưởng mảng hậu tố.

In [7]:
s = "aab"
suf = sorted(s[i:] for i in range(len(s)))
lcps = [lcp(suf[t - 1], suf[t]) for t in range(1, len(suf))]
print("hậu tố đã sắp:", suf, "| LCP liền kề:", lcps)
print("n(n+1)/2 - ΣLCP =", len(s) * (len(s) + 1) // 2 - sum(lcps))

hậu tố đã sắp: ['aab', 'ab', 'b'] | LCP liền kề: [1, 0]
n(n+1)/2 - ΣLCP = 5


## 5. Cách 1 — Brute Force / baseline: set mọi chuỗi con

**Ý tưởng:** đưa mọi `s[i:j]` vào `set`, trả về kích thước.

In [8]:
def count_distinct_set(s):
    n = len(s)
    return len({s[i:j] for i in range(n) for j in range(i + 1, n + 1)})

for s, want in examples:
    print(count_distinct_set(s), "kỳ vọng", want)

21 kỳ vọng 21
28 kỳ vọng 28


**Độ phức tạp:** có `~n²/2` chuỗi con, mỗi chuỗi tạo bằng slicing và băm tốn `O(độ dài)` ⇒ Time `O(n³)`, Space `O(n³)` ký tự trong set (worst case khi mọi chuỗi con khác nhau). Với `n = 500`: khoảng `2·10⁷` ký tự — Python vẫn chạy được, nhưng là giới hạn của bài.

**Nút thắt:** tạo bản sao cho từng chuỗi con. Trie và hash tránh tạo chuỗi mới.

In [9]:
random.seed(1)
for n in [100, 300, 500]:
    s = "".join(random.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(n))
    t0 = time.perf_counter(); r = count_distinct_set(s)
    print(f"n={n}: {r} chuỗi con phân biệt, {time.perf_counter() - t0:.3f}s")

n=100: 4964 chuỗi con phân biệt, 0.001s
n=300: 44816 chuỗi con phân biệt, 0.011s
n=500: 124620 chuỗi con phân biệt, 0.037s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** chuỗi con phân biệt ↔ **tiền tố phân biệt của các hậu tố**. Do đó:
- Trie hậu tố: số nút (trừ gốc) = đáp án.
- Mảng hậu tố: sắp hậu tố theo thứ tự từ điển; tiền tố chung của một hậu tố với **mọi** hậu tố đứng trước lớn nhất là với **hậu tố liền trước** ⇒ số tiền tố "mới" của hậu tố thứ `t` trong thứ tự = `len(suffix) − LCP(t−1, t)`. Tổng lại: `n(n+1)/2 − Σ LCP`.

**Insight 2 (hash):** với mỗi độ dài `L`, số chuỗi con phân biệt độ dài `L` = số giá trị hash phân biệt của các đoạn `s[i:i+L]` (giả sử không va chạm).

**Insight 3 (automaton):** suffix automaton là đồ thị nhỏ nhất (≤ `2n − 1` trạng thái) nhận đúng mọi chuỗi con của `s`. Mỗi trạng thái `v` đại diện cho các chuỗi con có độ dài trong `(len[link[v]], len[v]]` ⇒ đáp án = `Σ (len[v] − len[link[v]])`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Trie của mọi hậu tố

Từng đoạn:
1. Với mỗi `i`, đi từ gốc theo `s[i], s[i+1], …`.
2. Mỗi lần tạo nút mới ⇒ một chuỗi con mới `s[i..j]` ⇒ `cnt += 1`.

Cài bằng mảng `children[node][c]` (26 ô) nhanh hơn dict lồng nhau.

In [10]:
def count_distinct_trie(s):
    nxt = [[0] * 26]          # nút 0 là gốc; 0 nghĩa là chưa có con (gốc không bao giờ là con)
    cnt = 0
    for i in range(len(s)):
        cur = 0
        for j in range(i, len(s)):
            c = ord(s[j]) - 97
            if nxt[cur][c] == 0:
                nxt.append([0] * 26)
                nxt[cur][c] = len(nxt) - 1
                cnt += 1
            cur = nxt[cur][c]
    return cnt

for s, want in examples:
    print(count_distinct_trie(s), "kỳ vọng", want)

21 kỳ vọng 21
28 kỳ vọng 28


Code nộp LeetCode của Cách 2:

```python
class Solution:
    def countDistinct(self, s: str) -> int:
        nxt = [[0] * 26]
        cnt = 0
        for i in range(len(s)):
            cur = 0
            for j in range(i, len(s)):
                c = ord(s[j]) - 97
                if nxt[cur][c] == 0:
                    nxt.append([0] * 26)
                    nxt[cur][c] = len(nxt) - 1
                    cnt += 1
                cur = nxt[cur][c]
        return cnt
```

**Độ phức tạp:** Time `O(n²)`, Space `O(n² · 26)` worst case (mỗi nút 26 ô) — với `n = 500` là ~125k nút × 26 ≈ 3,3 triệu ô, chấp nhận được. Dùng `dict` cho mỗi nút thì Space `O(n²)`.

### 7.2 Cách 3 — Rolling hash theo từng độ dài

Từng đoạn:
1. Tính `h`, `p` (mục 3.4).
2. Với mỗi `L = 1..n`, đưa `sub_hash(i, i+L)` của mọi `i` vào một set, cộng kích thước set.

Gộp độ dài vào khóa (`(L, hash)`) cũng được, nhưng tách theo `L` giảm xác suất va chạm (chỉ so sánh chuỗi cùng độ dài).

In [11]:
def count_distinct_hash(s):
    n = len(s)
    h, p = prefix_hash(s)
    total = 0
    for L in range(1, n + 1):
        pL = p[L]
        seen = {(h[i + L] - h[i] * pL) % MOD for i in range(n - L + 1)}
        total += len(seen)
    return total

for s, want in examples:
    print(count_distinct_hash(s), "kỳ vọng", want)

21 kỳ vọng 21
28 kỳ vọng 28


**Độ phức tạp:** Time `O(n²)` (trung bình, thao tác set `O(1)`), Space `O(n)` cho mỗi set. Xác suất va chạm với `M ≈ 2.3·10¹⁸` và ≤ `n²/2 ≈ 1.25·10⁵` chuỗi mỗi lần: theo bài toán ngày sinh cỡ `n⁴/M ≈ 10⁻⁸` — rất nhỏ nhưng **không bằng 0**. Nếu cần chắc chắn, dùng Trie / mảng hậu tố.

### 7.3 Cách 4 — Mảng hậu tố + LCP (Kasai)

Từng đoạn:
1. **Mảng hậu tố** `sa`: chỉ số bắt đầu các hậu tố theo thứ tự từ điển. Với `n ≤ 500`, sắp xếp trực tiếp `sorted(range(n), key=lambda i: s[i:])` là đủ (`O(n² log n)`); bên dưới cài thêm thuật toán **nhân đôi** `O(n log² n)` để tham khảo.
2. **Kasai**: `rank[sa[t]] = t`; duyệt `i = 0..n−1` theo thứ tự trong chuỗi, giữ `k` = LCP hiện tại; khi chuyển từ `i` sang `i+1`, LCP giảm tối đa 1 ⇒ tổng `O(n)`.
3. Đáp án `n(n+1)/2 − Σ lcp`.

In [12]:
def suffix_array_doubling(s):
    n = len(s)
    rank = [ord(c) for c in s]
    sa = list(range(n))
    k = 1
    while True:
        key = lambda i: (rank[i], rank[i + k] if i + k < n else -1)
        sa.sort(key=key)
        new = [0] * n
        for t in range(1, n):
            new[sa[t]] = new[sa[t - 1]] + (key(sa[t]) != key(sa[t - 1]))
        rank = new
        if rank[sa[-1]] == n - 1:
            break
        k <<= 1
    return sa

def kasai(s, sa):
    n = len(s); rank = [0] * n
    for t, i in enumerate(sa):
        rank[i] = t
    lcp_arr = [0] * n           # lcp_arr[t] = LCP(sa[t-1], sa[t]); lcp_arr[0] = 0
    k = 0
    for i in range(n):
        if rank[i] == 0:
            k = 0; continue
        j = sa[rank[i] - 1]
        while i + k < n and j + k < n and s[i + k] == s[j + k]:
            k += 1
        lcp_arr[rank[i]] = k
        if k:
            k -= 1
    return lcp_arr

def count_distinct_sa(s):
    n = len(s)
    sa = suffix_array_doubling(s)
    return n * (n + 1) // 2 - sum(kasai(s, sa))

for s, want in examples:
    print(count_distinct_sa(s), "kỳ vọng", want)
print("SA bằng nhân đôi khớp sorted:", suffix_array_doubling("aabbaba") == sorted(range(7), key=lambda i: "aabbaba"[i:]))

21 kỳ vọng 21
28 kỳ vọng 28
SA bằng nhân đôi khớp sorted: True


**Độ phức tạp:** SA nhân đôi `O(n log² n)` (mỗi vòng sắp xếp `O(n log n)`, `log n` vòng), Kasai `O(n)` ⇒ Time `O(n log² n)`, Space `O(n)`.

### 7.4 Cách 5 — Suffix automaton `O(n)` (follow-up)

Suffix automaton (SAM) xây **trực tuyến**, thêm từng ký tự. Mỗi trạng thái có `len` (độ dài chuỗi dài nhất của lớp), `link` (suffix link), `next` (chuyển tiếp). Thủ tục thêm ký tự `c`:
1. Tạo trạng thái `cur` với `len = len[last] + 1`.
2. Đi theo suffix link từ `last`, thêm cạnh `c → cur` cho tới khi gặp trạng thái `p` đã có cạnh `c`.
3. Nếu không gặp: `link[cur] = 0`. Nếu `q = next[p][c]` có `len[q] == len[p] + 1`: `link[cur] = q`. Ngược lại **nhân bản** `q` thành `clone` với `len = len[p] + 1`, chuyển hướng các cạnh `c` trỏ vào `q` sang `clone`.

Đáp án `Σ_{v ≠ 0} (len[v] − len[link[v]])`.

In [13]:
def count_distinct_sam(s):
    length = [0]; link = [-1]; nxt = [{}]
    last = 0
    for ch in s:
        cur = len(length)
        length.append(length[last] + 1); link.append(-1); nxt.append({})
        p = last
        while p != -1 and ch not in nxt[p]:
            nxt[p][ch] = cur
            p = link[p]
        if p == -1:
            link[cur] = 0
        else:
            q = nxt[p][ch]
            if length[p] + 1 == length[q]:
                link[cur] = q
            else:
                clone = len(length)
                length.append(length[p] + 1); link.append(link[q]); nxt.append(dict(nxt[q]))
                while p != -1 and nxt[p].get(ch) == q:
                    nxt[p][ch] = clone
                    p = link[p]
                link[q] = link[cur] = clone
        last = cur
    return sum(length[v] - length[link[v]] for v in range(1, len(length)))

for s, want in examples:
    print(count_distinct_sam(s), "kỳ vọng", want)

21 kỳ vọng 21
28 kỳ vọng 28


**Độ phức tạp:** Time `O(n · |Σ|)` với cách sao chép dict khi clone, thực tế `O(n)` khấu hao với bảng chữ cái cố định; Space `O(n · |Σ|)`. Số trạng thái ≤ `2n − 1`, số cạnh ≤ `3n − 4`.

## 8. TRACE từng bước

Trace Cách 2 (Trie) trên `s = "aabbaba"` — với mỗi hậu tố, in các chuỗi con **mới** được tạo:

In [14]:
def trace_trie(s):
    root = {}; total = 0
    for i in range(len(s)):
        cur = root; new = []
        for j in range(i, len(s)):
            ch = s[j]
            if ch not in cur:
                cur[ch] = {}; new.append(s[i:j + 1])
            cur = cur[ch]
        total += len(new)
        print(f"hậu tố {s[i:]!r:10}: +{len(new)} mới {new}  (tổng {total})")
    return total

trace_trie("aabbaba")

hậu tố 'aabbaba' : +7 mới ['a', 'aa', 'aab', 'aabb', 'aabba', 'aabbab', 'aabbaba']  (tổng 7)
hậu tố 'abbaba'  : +5 mới ['ab', 'abb', 'abba', 'abbab', 'abbaba']  (tổng 12)
hậu tố 'bbaba'   : +5 mới ['b', 'bb', 'bba', 'bbab', 'bbaba']  (tổng 17)
hậu tố 'baba'    : +3 mới ['ba', 'bab', 'baba']  (tổng 20)
hậu tố 'aba'     : +1 mới ['aba']  (tổng 21)
hậu tố 'ba'      : +0 mới []  (tổng 21)
hậu tố 'a'       : +0 mới []  (tổng 21)


21

Bảng mảng hậu tố + LCP cho `"aabbaba"`:

In [15]:
def trace_sa(s):
    sa = suffix_array_doubling(s); lc = kasai(s, sa); n = len(s)
    print(f"{'t':>2} {'sa':>3} {'hậu tố':10} {'LCP':>3} {'mới = len - LCP':>16}")
    for t, i in enumerate(sa):
        print(f"{t:>2} {i:>3} {s[i:]:10} {lc[t]:>3} {n - i - lc[t]:>16}")
    print("tổng =", n * (n + 1) // 2, "-", sum(lc), "=", n * (n + 1) // 2 - sum(lc))

trace_sa("aabbaba")

 t  sa hậu tố     LCP  mới = len - LCP
 0   6 a            0                1
 1   0 aabbaba      1                6
 2   4 aba          1                2
 3   1 abbaba       2                4
 4   5 ba           0                2
 5   3 baba         2                2
 6   2 bbaba        1                4
tổng = 28 - 7 = 21


In [16]:
def trace_sam(s):
    length = [0]; link = [-1]; nxt = [{}]; last = 0
    for ch in s:
        cur = len(length); length.append(length[last] + 1); link.append(-1); nxt.append({})
        p = last
        while p != -1 and ch not in nxt[p]:
            nxt[p][ch] = cur; p = link[p]
        if p == -1:
            link[cur] = 0
        else:
            q = nxt[p][ch]
            if length[p] + 1 == length[q]:
                link[cur] = q
            else:
                clone = len(length); length.append(length[p] + 1); link.append(link[q]); nxt.append(dict(nxt[q]))
                while p != -1 and nxt[p].get(ch) == q:
                    nxt[p][ch] = clone; p = link[p]
                link[q] = link[cur] = clone
        last = cur
        cnt = sum(length[v] - length[link[v]] for v in range(1, len(length)))
        print(f"thêm {ch!r}: số trạng thái = {len(length):2d}, số chuỗi con phân biệt của tiền tố = {cnt}")

trace_sam("aabbaba")

thêm 'a': số trạng thái =  2, số chuỗi con phân biệt của tiền tố = 1
thêm 'a': số trạng thái =  3, số chuỗi con phân biệt của tiền tố = 2
thêm 'b': số trạng thái =  4, số chuỗi con phân biệt của tiền tố = 5
thêm 'b': số trạng thái =  6, số chuỗi con phân biệt của tiền tố = 8
thêm 'a': số trạng thái =  7, số chuỗi con phân biệt của tiền tố = 12
thêm 'b': số trạng thái =  9, số chuỗi con phân biệt của tiền tố = 16
thêm 'a': số trạng thái = 11, số chuỗi con phân biệt của tiền tố = 21


## 9. Code nộp LeetCode

Với `n ≤ 500`, Trie `O(n²)` là lựa chọn chắc chắn (không va chạm, dễ viết). Bản dưới dùng dict cho mỗi nút để tiết kiệm bộ nhớ.

In [17]:
class Solution:
    def countDistinct(self, s: str) -> int:
        root = {}
        cnt = 0
        for i in range(len(s)):
            cur = root
            for j in range(i, len(s)):
                ch = s[j]
                nxt = cur.get(ch)
                if nxt is None:
                    nxt = cur[ch] = {}
                    cnt += 1
                cur = nxt
        return cnt

sol = Solution()
for s, want in examples:
    got = sol.countDistinct(s)
    print(got, "OK" if got == want else "SAI")

21 OK
28 OK


## 10. Vì sao thuật toán đúng

**Trie.** Ánh xạ "nút ↦ chuỗi trên đường đi từ gốc" là song ánh giữa các nút (trừ gốc) và tập các tiền tố của các hậu tố đã chèn. Tập đó chính là tập chuỗi con (mọi `s[i:j]` là tiền tố của `s[i:]`, và ngược lại mọi tiền tố của hậu tố là chuỗi con). Mỗi nút được tạo đúng một lần ⇒ `cnt` = số chuỗi con phân biệt.

**Mảng hậu tố.** Sắp các hậu tố `S_0 < S_1 < … < S_{n−1}`. Một tiền tố độ dài `ℓ` của `S_t` đã xuất hiện ở một hậu tố đứng trước ⇔ `ℓ ≤ max_{u<t} LCP(S_u, S_t) = LCP(S_{t−1}, S_t)` (tính chất thứ tự từ điển: LCP với hậu tố xa hơn không lớn hơn LCP với hậu tố liền kề). Vậy `S_t` đóng góp đúng `|S_t| − LCP(S_{t−1}, S_t)` tiền tố mới. Cộng lại: `Σ|S_t| − ΣLCP = n(n+1)/2 − ΣLCP`.

**Kasai đúng** nhờ bổ đề: nếu LCP của hậu tố `i` với hậu tố liền trước nó trong SA là `k > 0`, thì LCP của hậu tố `i+1` với hậu tố liền trước nó ≥ `k − 1`.

**Hash.** Đúng nếu không va chạm; với hai đoạn cùng độ dài khác nhau, xác suất trùng hash (theo `B` ngẫu nhiên) ≤ `L/M`.

**SAM.** Mỗi chuỗi con của `s` tương ứng với đúng một đường đi từ trạng thái đầu; các chuỗi kết thúc tại trạng thái `v` có độ dài đúng bằng các số nguyên trong `(len[link[v]], len[v]]` ⇒ đếm theo trạng thái.

## 11. Phân tích Time / Space

Gọi `n = len(s) ≤ 500`, `Σ = 26`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Set chuỗi con | `O(n³)` | `O(n³)` | tạo bản sao mọi chuỗi con |
| 2. Trie hậu tố | `O(n²)` | `O(n²)` nút | chắc chắn đúng |
| 3. Rolling hash | `O(n²)` trung bình | `O(n)` | rủi ro va chạm rất nhỏ |
| 4. SA + LCP | `O(n log² n)` | `O(n)` | nhân đôi + Kasai |
| 5. Suffix automaton | `O(n·Σ)` | `O(n·Σ)` | trả lời follow-up |

In [18]:
random.seed(5)
s_big = "".join(random.choice("ab") for _ in range(500))      # bảng chữ cái nhỏ -> nhiều trùng lặp
s_big2 = "".join(random.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(500))
for name, fn in [("set", count_distinct_set), ("trie", count_distinct_trie), ("hash", count_distinct_hash),
                 ("SA", count_distinct_sa), ("SAM", count_distinct_sam), ("submit", Solution().countDistinct)]:
    t0 = time.perf_counter(); r1 = fn(s_big); r2 = fn(s_big2)
    print(f"{name:6s}: {r1:6d} {r2:6d}  {time.perf_counter() - t0:.3f}s")

s_huge = "".join(random.choice("abc") for _ in range(10**5))
t0 = time.perf_counter(); r = count_distinct_sam(s_huge)
print(f"SAM với n = 100000: {r}  {time.perf_counter() - t0:.3f}s")

set   : 121259 124656  0.067s


trie  : 121259 124656  0.123s


hash  : 121259 124656  0.047s
SA    : 121259 124656  0.001s
SAM   : 121259 124656  0.001s


submit: 121259 124656  0.056s


SAM với n = 100000: 4999090265  0.133s


## 12. Bộ test

In [19]:
fns = [count_distinct_set, count_distinct_trie, count_distinct_hash, count_distinct_sa,
       count_distinct_sam, lambda s: Solution().countDistinct(s)]

for s, want in examples:
    for fn in fns:
        assert fn(s) == want

edge = [("a", 1), ("aa", 2), ("aaaa", 4), ("ab", 3), ("aba", 5), ("abab", 7),
        ("z" * 500, 500), ("abcdefghijklmnopqrstuvwxyz", 351)]
for s, want in edge:
    assert count_distinct_set(s) == want, s
    for fn in fns:
        assert fn(s) == want, (fn, s)

random.seed(2024)
for _ in range(600):
    n = random.randint(1, 40)
    alpha = random.choice(["a", "ab", "abc", "abcdefghijklmnopqrstuvwxyz"])
    s = "".join(random.choice(alpha) for _ in range(n))
    want = count_distinct_set(s)
    for fn in fns[1:]:
        assert fn(s) == want, (fn, s)
print("ví dụ + edge + 600 chuỗi ngẫu nhiên: OK")

ví dụ + edge + 600 chuỗi ngẫu nhiên: OK


In [20]:
# kiểm tra thêm SA nhân đôi và Kasai riêng lẻ so với cách ngây thơ
random.seed(7)
for _ in range(300):
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 60)))
    sa = suffix_array_doubling(s)
    assert sa == sorted(range(len(s)), key=lambda i: s[i:])
    lc = kasai(s, sa)
    assert lc == [0] + [lcp(s[sa[t - 1]:], s[sa[t]:]) for t in range(1, len(s))]
print("SA nhân đôi + Kasai khớp cài đặt ngây thơ")

SA nhân đôi + Kasai khớp cài đặt ngây thơ


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đếm `n(n+1)/2` mà quên chuỗi con trùng nhau

In [21]:
def wrong_formula(s):
    n = len(s)
    return n * (n + 1) // 2

for s, want in examples:
    print(f"{s!r}: sai = {wrong_formula(s)}, đúng = {want}")

'aabbaba': sai = 28, đúng = 21
'abcdefg': sai = 28, đúng = 28


Đúng với `"abcdefg"` (mọi ký tự khác nhau) nhưng sai với `"aabbaba"`. **Sửa:** phải trừ phần trùng (Σ LCP) hoặc đếm bằng cấu trúc loại trùng.

### Lỗi 2 — Rolling hash không tách theo độ dài và dùng modulo nhỏ

Modulo nhỏ (ví dụ `10⁹+7` với base cố định nhỏ) và trộn chung mọi độ dài ⇒ va chạm dễ xảy ra. Minh họa với modulo cố tình rất nhỏ:

In [22]:
def wrong_small_mod(s, M=97, B=31):
    n = len(s); seen = set()
    for i in range(n):
        hv = 0
        for j in range(i, n):
            hv = (hv * B + ord(s[j])) % M
            seen.add(hv)                     # SAI: không kèm độ dài, M quá nhỏ
    return len(seen)

s = "aabbaba"
print("sai =", wrong_small_mod(s), "| đúng =", count_distinct_set(s))
s = "abcdefghijklmnopqrst"
print("sai =", wrong_small_mod(s), "| đúng =", count_distinct_set(s))

sai = 9 | đúng = 21
sai = 82 | đúng = 210


**Sửa:** modulo lớn (`2⁶¹−1`) + base ngẫu nhiên, và tách theo độ dài (hoặc dùng khóa `(L, hash)`). Khi cần đúng tuyệt đối, dùng Trie/SA.

### Lỗi 3 — Trie: đếm số **lần đi qua** nút thay vì số nút **được tạo**

In [23]:
def wrong_trie(s):
    root = {}; cnt = 0
    for i in range(len(s)):
        cur = root
        for j in range(i, len(s)):
            cur = cur.setdefault(s[j], {})
            cnt += 1                        # SAI: đếm cả nút đã có
    return cnt

print("sai =", wrong_trie("aabbaba"), "| đúng =", count_distinct_trie("aabbaba"))

sai = 28 | đúng = 21


**Sửa:** chỉ tăng `cnt` khi tạo nút mới.

### Lỗi 4 — Kasai quên giảm `k` (hoặc reset `k` về 0 mỗi vòng)

Reset `k = 0` mỗi vòng vẫn đúng nhưng mất tính `O(n)` (thành `O(n²)`); **không giảm** `k` khi chuyển sang `i+1` thì sai, vì LCP có thể nhỏ hơn.

In [24]:
def wrong_kasai(s, sa):
    n = len(s); rank = [0] * n
    for t, i in enumerate(sa): rank[i] = t
    out = [0] * n; k = 0
    for i in range(n):
        if rank[i] == 0:
            k = 0; continue
        j = sa[rank[i] - 1]
        while i + k < n and j + k < n and s[i + k] == s[j + k]:
            k += 1
        out[rank[i]] = k                   # SAI: thiếu k -= 1, k chỉ tăng
    return out

s = "aabbaba"; sa = suffix_array_doubling(s)
print("LCP sai :", wrong_kasai(s, sa))
print("LCP đúng:", kasai(s, sa))

LCP sai : [0, 1, 2, 2, 2, 2, 2]
LCP đúng: [0, 1, 1, 2, 0, 2, 1]


Thiếu `k −= 1`, `k` mang nguyên giá trị cũ sang hậu tố mới — vòng `while` không kiểm tra lại `k` ký tự đầu nên LCP bị **phóng đại**. **Sửa:** `if k: k -= 1` sau mỗi bước.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm/so sánh **chuỗi con phân biệt**" ⇒ nghĩ ngay: chuỗi con = tiền tố của hậu tố ⇒ Trie hậu tố / SA+LCP / SAM.
- `n ≤ 500–2000` ⇒ `O(n²)` Trie hoặc hash là đủ; `n ≥ 10⁵` ⇒ SA hoặc SAM.
- Hash: luôn dùng modulo lớn + base ngẫu nhiên; với Python, `2⁶¹−1` không tràn vì số nguyên tùy ý.
- Công thức vàng: `#distinct substrings = n(n+1)/2 − Σ LCP`.
- SAM: `Σ (len[v] − len[link[v]])` — nhớ khung `extend(c)` là dùng được cho nhiều bài (LCS hai chuỗi, đếm lần xuất hiện, …).

## 15. Biến thể của bài

1. **Đếm chuỗi con phân biệt theo từng tiền tố** (sau mỗi ký tự thêm vào) — SAM trực tuyến làm được (mục 8 đã in).
2. **Chuỗi con phân biệt có độ dài đúng `L`** — hash theo `L` (Cách 3, chỉ một vòng).
3. **Chuỗi con lặp lại dài nhất** (LeetCode 1044) — `max(LCP)` trên SA, hoặc binary search + hash.
4. **Chuỗi con chung dài nhất của hai chuỗi** — SA của `a#b` hoặc SAM.
5. **Đếm chuỗi con phân biệt là palindrome** — palindromic tree (eertree).
6. **Đếm chuỗi con xuất hiện ít nhất 2 lần** — trên SA: `Σ` số LCP "mới" … hoặc trên SAM với `occ ≥ 2`.

In [25]:
def longest_repeated_substring(s):
    """Biến thể 3: dùng SA + LCP."""
    if len(s) < 2: return ""
    sa = suffix_array_doubling(s); lc = kasai(s, sa)
    t = max(range(len(s)), key=lambda t: lc[t])
    return s[sa[t]:sa[t] + lc[t]]

def distinct_of_length(s, L):
    """Biến thể 2."""
    h, p = prefix_hash(s)
    return len({sub_hash(h, p, i, i + L) for i in range(len(s) - L + 1)})

print("chuỗi lặp dài nhất của 'banana':", repr(longest_repeated_substring("banana")))
print("số chuỗi con độ dài 2 của 'aabbaba':", distinct_of_length("aabbaba", 2), "(aa, ab, bb, ba)")

chuỗi lặp dài nhất của 'banana': 'ana'
số chuỗi con độ dài 2 của 'aabbaba': 4 (aa, ab, bb, ba)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Set | đưa mọi chuỗi con vào set | `O(n³)` | `O(n³)` | kiểm chứng, `n ≤ 500` |
| 2. Trie hậu tố | đếm nút | `O(n²)` | `O(n²)` | **nộp bài** với `n ≤ 500` |
| 3. Rolling hash | set hash theo độ dài | `O(n²)` | `O(n)` | khi bộ nhớ hạn chế |
| 4. SA + LCP | `n(n+1)/2 − ΣLCP` | `O(n log² n)` | `O(n)` | `n` lớn, cần thêm truy vấn LCP |
| 5. SAM | `Σ len − len[link]` | `O(n)` | `O(n·Σ)` | follow-up `O(n)`, trực tuyến |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `countDistinct("aaa")` và `countDistinct("abab")`?

**Bài 2 (tính tay):** liệt kê hậu tố đã sắp và LCP của `"abab"`, rồi áp công thức.

**Bài 3 (điền TODO):** hoàn thiện `distinct_with_dict_trie(s)` dùng `dict` lồng nhau nhưng **chỉ chèn hậu tố có độ dài ≤ m** (đếm chuỗi con phân biệt có độ dài ≤ m).

**Bài 4 (tự viết lại):** viết lại Cách 4 với SA tạo bằng `sorted(range(n), key=lambda i: s[i:])` và LCP tính trực tiếp bằng hàm `lcp` (không Kasai). So sánh tốc độ với `n = 500`.

**Bài 5 (mini challenge):** đếm số chuỗi con phân biệt **xuất hiện ít nhất hai lần** trong `s`. Gợi ý: dùng SA + LCP — một chuỗi con lặp lại tương ứng với một tiền tố chung của hai hậu tố liền kề. Kiểm chứng bằng `Counter`.

In [26]:
def distinct_with_dict_trie(s, m):
    # TODO: với mỗi i, đi tối đa m ký tự từ s[i], đếm nút mới
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `"aaa"` chỉ có `a, aa, aaa`. (2) `"abab"`: `a, b, ab, ba, aba, bab, abab`. (3) `3` và `7`.

**Bài 2:** (1) Hậu tố: `abab, bab, ab, b`. (2) Sắp: `ab, abab, b, bab`; LCP liền kề `2, 0, 1`. (3) `10 − 3 = 7`.

**Bài 3:** (1) Giống Trie, vòng `j` chạy tới `min(n, i + m)`. (2) Đếm nút mới. (3) Kiểm tra với `set` chuỗi con độ dài ≤ m.

**Bài 4:** (1) `sa = sorted(...)`. (2) `sum(lcp(s[sa[t-1]:], s[sa[t]:]) ...)`. (3) Đo thời gian bằng `time.perf_counter`.

**Bài 5:** (1) Chuỗi con `x` xuất hiện ≥ 2 lần ⇔ `x` là tiền tố chung của hai hậu tố ⇔ là tiền tố của hậu tố `sa[t]` với độ dài ≤ `lcp[t]` cho một `t` nào đó. (2) Đếm phân biệt: mỗi `t` đóng góp `max(0, lcp[t] − lcp[t−1])`. (3) Đáp án `Σ max(0, lcp[t] − lcp[t−1])`.

## 19. Đáp án tham khảo

In [27]:
from collections import Counter

# Bài 1
assert Solution().countDistinct("aaa") == 3 and Solution().countDistinct("abab") == 7

# Bài 2
s = "abab"; sa = sorted(range(4), key=lambda i: s[i:])
assert [s[i:] for i in sa] == ["ab", "abab", "b", "bab"] and kasai(s, sa) == [0, 2, 0, 1]

# Bài 3
def distinct_with_dict_trie_ans(s, m):
    root = {}; cnt = 0
    for i in range(len(s)):
        cur = root
        for j in range(i, min(len(s), i + m)):
            if s[j] not in cur:
                cur[s[j]] = {}; cnt += 1
            cur = cur[s[j]]
    return cnt

random.seed(31)
for _ in range(300):
    s = "".join(random.choice("abc") for _ in range(random.randint(1, 30))); m = random.randint(1, 10)
    want = len({s[i:j] for i in range(len(s)) for j in range(i + 1, min(len(s), i + m) + 1)})
    assert distinct_with_dict_trie_ans(s, m) == want

# Bài 4
def count_distinct_sa_naive(s):
    n = len(s); sa = sorted(range(n), key=lambda i: s[i:])
    return n * (n + 1) // 2 - sum(lcp(s[sa[t - 1]:], s[sa[t]:]) for t in range(1, n))

random.seed(32)
s5 = "".join(random.choice("ab") for _ in range(500))
t0 = time.perf_counter(); a = count_distinct_sa_naive(s5); t1 = time.perf_counter(); b = count_distinct_sa(s5); t2 = time.perf_counter()
assert a == b == count_distinct_trie(s5)
print(f"Bài 4: SA ngây thơ {t1 - t0:.3f}s | SA nhân đôi + Kasai {t2 - t1:.3f}s")
print("Bài 1-4: OK")

Bài 4: SA ngây thơ 0.001s | SA nhân đôi + Kasai 0.001s
Bài 1-4: OK


In [28]:
# Bài 5 — chuỗi con phân biệt xuất hiện ít nhất 2 lần
def count_repeated_distinct(s):
    sa = suffix_array_doubling(s); lc = kasai(s, sa)
    return sum(max(0, lc[t] - lc[t - 1]) for t in range(1, len(s)))

def brute_repeated(s):
    c = Counter(s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1))
    return sum(1 for v in c.values() if v >= 2)

assert count_repeated_distinct("aabbaba") == brute_repeated("aabbaba")
random.seed(33)
for _ in range(400):
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 35)))
    assert count_repeated_distinct(s) == brute_repeated(s), s
print("Bài 5: OK; 'aabbaba' có", count_repeated_distinct("aabbaba"), "chuỗi con lặp lại phân biệt")

Bài 5: OK; 'aabbaba' có 4 chuỗi con lặp lại phân biệt


## 20. Template nhớ nhanh + Checklist

```python
# Trie hậu tố — đếm chuỗi con phân biệt (n ≤ vài nghìn)
root, cnt = {}, 0
for i in range(n):
    cur = root
    for j in range(i, n):
        if s[j] not in cur:
            cur[s[j]] = {}; cnt += 1
        cur = cur[s[j]]

# SA + LCP:  answer = n*(n+1)//2 - sum(lcp)
# SAM:       answer = sum(len[v] - len[link[v]] for v != root)
```

**Checklist trước khi submit:**
- [ ] Không tính chuỗi rỗng.
- [ ] Trie: chỉ đếm khi **tạo** nút mới.
- [ ] Hash: modulo lớn, base ngẫu nhiên, tách theo độ dài.
- [ ] Kasai: `k −= 1` sau mỗi bước, `k = 0` khi `rank[i] == 0`.
- [ ] SAM: nhớ nhân bản (clone) khi `len[p] + 1 != len[q]`.
- [ ] Thử `"a"`, `"aaaa"`, chuỗi toàn ký tự khác nhau.